# Forecasting Showdown — Results

Reads all runs from the local MLflow experiment `forecasting-showdown` and renders
a comparison table sorted by MAE.

**Pre-requisite**: run `uv run python scripts/run_all.py` at least once.

In [ ]:
import matplotlib.pyplot as plt
import mlflow
import pandas as pd

from src.visuals.charts import forecast_overlay, mae_bar, metrics_grid, train_time_scatter

EXPERIMENT = "forecasting-showdown"
METRIC_COLS = ["mae", "rmse", "mape", "smape", "latency_s", "train_time_s"]

In [ ]:
client = mlflow.tracking.MlflowClient()

exp = client.get_experiment_by_name(EXPERIMENT)
if exp is None:
    raise RuntimeError(f"Experiment '{EXPERIMENT}' not found — run scripts/run_all.py first.")

all_runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["start_time DESC"],  # newest first so dedup keeps latest
)
print(f"Found {len(all_runs)} total runs in experiment '{EXPERIMENT}'")

# Keep only the most recent *successful* run per model.
# Runs killed mid-training have no mae metric and are skipped.
seen: set[str] = set()
runs = []
for run in all_runs:
    model_name = run.data.tags.get("model", run.info.run_name)
    if model_name in seen:
        continue
    if run.data.metrics.get("mae") is None:
        continue  # killed / partial run — skip
    seen.add(model_name)
    runs.append(run)

print(f"Using {len(runs)} models with complete results")

In [ ]:
rows = []
for run in runs:
    row = {"model": run.data.tags.get("model", run.info.run_name)}
    for col in METRIC_COLS:
        row[col] = run.data.metrics.get(col)
    rows.append(row)

if not rows:
    raise RuntimeError("No completed runs found — run scripts/run_all.py first.")

df = (
    pd.DataFrame(rows)
    .sort_values("mae")          # safe: None rows filtered out above
    .reset_index(drop=True)
)
df.index = range(1, len(df) + 1)  # rank from 1
df

In [ ]:
float_cols = [c for c in METRIC_COLS if c in df.columns]
df.style \
    .format({c: "{:.4f}" for c in float_cols}) \
    .background_gradient(subset=["mae"], cmap="RdYlGn_r") \
    .set_caption("All models — sorted by MAE (lower is better)") \
    .set_table_styles([{"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold")]}])

## MAE comparison (bar chart)

In [ ]:
fig = mae_bar(df)
plt.show()

## Error metrics overview

In [ ]:
fig = metrics_grid(df)
plt.show()

## Train time vs MAE (efficiency scatter)

In [ ]:
fig = train_time_scatter(df)
plt.show()

In [ ]:
from src.config import load_config
from src.data.features import build_features
from src.data.loader import load_raw
from src.data.splits import chronological_split
from src.models.ensemble import EnsembleForecaster
from src.models.lgbm_model import LGBMForecaster
from src.models.naive import NaiveForecaster
from src.utils import empty_x

# ── Load data (same split used in the benchmark run) ──────────────────────
_df = load_raw("../data/energy.csv")
_df = build_features(_df, freq="h")
_train, _val, _test = chronological_split(_df)

_target = "demand"
_feat_cols = [c for c in _df.columns if c != _target]

X_train, y_train = _train[_feat_cols], _train[_target]
X_test,  y_test  = _test[_feat_cols],  _test[_target]

# ── Fit models ─────────────────────────────────────────────────────────────
print("Fitting ensemble ...", end=" ", flush=True)
m_ens = EnsembleForecaster(load_config("ensemble"))
m_ens.fit(X_train, y_train)
print("done")

print("Fitting lgbm    ...", end=" ", flush=True)
m_lgbm = LGBMForecaster(load_config("lgbm"))
m_lgbm.fit(X_train, y_train)
print("done")

print("Fitting naive   ...", end=" ", flush=True)
m_naive = NaiveForecaster(load_config("naive"))
m_naive.fit(empty_x(y_train.index), y_train)
print("done")

# ── Predict ────────────────────────────────────────────────────────────────
preds = {
    "ensemble": m_ens.predict(X_test),
    "lgbm":     m_lgbm.predict(X_test),
    "naive":    m_naive.predict(empty_x(y_test.index)),
}

fig = forecast_overlay(y_test, preds)
plt.show()

## Forecast overlay (best models vs actuals)

Re-fits the three fastest/best models on the training set and overlays their
predictions against the actual test-set demand for the first week (168 hours).
Models chosen: **ensemble** (best MAE), **lgbm** (best solo tabular), **naive** (baseline).